# Homework 4 - Reinforcement Learning

In [ ]:
import numpy as np
from typing import List, Optional, Tuple, Any, Iterable
import util_rl as util

## Problem 3a: Value Iteration on Number Line MDP

In [ ]:
def value_iteration(
        transitions: np.ndarray,
        rewards: np.ndarray,
        discount: float,
        epsilon: float = 0.001,
        valid_actions: Optional[np.ndarray] = None,
        state_ids: Optional[Iterable[Any]] = None,
        action_ids: Optional[Iterable[Any]] = None,
    ):
    transitions = np.asarray(transitions, dtype=np.float64)
    rewards = np.asarray(rewards, dtype=np.float64)
    num_states, num_actions, next_state_dim = transitions.shape

    if valid_actions is None:
        action_mask = np.ones((num_states, num_actions), dtype=bool)
    else:
        action_mask = np.asarray(valid_actions, dtype=bool)

    if state_ids is None:
        state_ids = np.arange(num_states)
    else:
        state_ids = np.array(list(state_ids), dtype=object)

    if action_ids is None:
        action_ids = np.arange(num_actions)
    else:
        action_ids = np.array(list(action_ids), dtype=object)

    tie_breaker = (np.arange(num_actions, dtype=np.float64) * 1e-12)[np.newaxis, :]

    def compute_q(v: np.ndarray) -> np.ndarray:
        """Q(s,a) = sum_s' T(s,a,s') * [R(s,a,s') + discount * V(s')]"""
        q = np.sum(transitions * (rewards + discount * v[np.newaxis, np.newaxis, :]), axis=2)
        return q

    def compute_policy(q: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
        """Extract best actions/values. Ties broken by largest action index."""
        masked_q = np.where(action_mask, q + tie_breaker, -np.inf)
        best_actions = np.argmax(masked_q, axis=1)
        best_values = q[np.arange(num_states), best_actions]
        best_values[~action_mask.any(axis=1)] = 0.0
        return best_actions, best_values

    print('Running value iteration...')

    terminal_mask = ~action_mask.any(axis=1)
    v = np.zeros(num_states, dtype=np.float64)

    while True:
        q = compute_q(v)
        best_actions, new_v = compute_policy(q)
        new_v[terminal_mask] = 0.0

        if np.max(np.abs(new_v - v)) < epsilon:
            v = new_v
            break
        v = new_v

    policy = np.full(num_states, None, dtype=object)
    for i in range(num_states):
        if action_mask[i].any():
            policy[i] = action_ids[best_actions[i]]

    return policy

In [ ]:
def run_vi_over_number_line(mdp: util.NumberLineMDP):
    num_states = mdp.num_states
    actions = np.array(list(mdp.actions), dtype=object)
    transitions = np.zeros((num_states, len(actions), num_states), dtype=np.float64)
    rewards = np.zeros_like(transitions)
    valid_actions = np.zeros((num_states, len(actions)), dtype=bool)
    for state_idx, state in enumerate(mdp.indexer.all_states()):
        if state in mdp.terminal_states:
            continue
        valid_actions[state_idx, :] = True
        for action_idx, action in enumerate(actions):
            forward_prob = 0.2 if action == 1 else 0.3
            backward_prob = 1.0 - forward_prob
            forward_state = state + 1
            backward_state = state - 1
            forward_reward = mdp.right_reward if forward_state == mdp.n else (
                mdp.left_reward if forward_state == -mdp.n else mdp.penalty)
            backward_reward = mdp.right_reward if backward_state == mdp.n else (
                mdp.left_reward if backward_state == -mdp.n else mdp.penalty)
            forward_idx = mdp.state_to_index(forward_state)
            backward_idx = mdp.state_to_index(backward_state)
            transitions[state_idx, action_idx, forward_idx] += forward_prob
            rewards[state_idx, action_idx, forward_idx] = forward_reward
            transitions[state_idx, action_idx, backward_idx] += backward_prob
            rewards[state_idx, action_idx, backward_idx] = backward_reward

    state_ids = np.array([mdp.index_to_state(i) for i in range(num_states)], dtype=object)
    pi = value_iteration(
        transitions,
        rewards,
        mdp.discount,
        valid_actions=valid_actions,
        state_ids=state_ids,
        action_ids=actions,
    )
    return state_ids, pi

In [ ]:
mdp = util.NumberLineMDP()
state_ids, pi = run_vi_over_number_line(mdp)

print("\nOptimal policy:")
for state, action in zip(state_ids, pi):
    print(f"  State {state:+d}: action {action}")

In [ ]:
import math, random
from typing import Callable

## Problem 3b: Model-Based Monte Carlo

In [ ]:
class ModelBasedMonteCarlo(util.RLAlgorithm):
    def __init__(
            self,
            actions: List,
            discount: float,
            num_states: int,
            state_to_index: Callable,
            index_to_state: Optional[Callable] = None,
            calc_val_iter_every: int = 10000,
            exploration_prob: float = 0.2,
        ) -> None:
        self.actions = list(actions)
        self.discount = discount
        self.num_states = int(num_states)
        self.state_to_index = state_to_index
        self.index_to_state = index_to_state or (lambda idx: idx)
        self.calc_val_iter_every = int(calc_val_iter_every)
        self.exploration_prob = exploration_prob
        self.num_iters = 0

        self.num_actions = len(self.actions)
        self.actions_array = np.array(self.actions, dtype=object)
        self.state_ids = np.array([self.index_to_state(i) for i in range(self.num_states)], dtype=object)

        self.transition_counts = np.zeros((self.num_states, self.num_actions, self.num_states), dtype=np.float64)
        self.reward_sums = np.zeros_like(self.transition_counts)
        self.valid_actions = np.zeros((self.num_states, self.num_actions), dtype=bool)

        self.pi_actions = np.full(self.num_states, None, dtype=object)
        self.pi_indices = np.full(self.num_states, -1, dtype=int)

    def _sync_policy_indices(self):
        self.pi_indices[:] = -1
        valid_mask = self.pi_actions != None
        for idx in np.where(valid_mask)[0]:
            self.pi_indices[idx] = int(np.argmax(self.actions_array == self.pi_actions[idx]))

    def get_action(self, state, explore: bool = True):
        if explore:
            self.num_iters += 1
        exploration_prob = self.exploration_prob
        if self.num_iters < 2e4:
            exploration_prob = 1.0
        elif self.num_iters > 1e6:
            exploration_prob = exploration_prob / math.log(self.num_iters - 100000 + 1)
        state_idx = int(self.state_to_index(state))
        policy_idx = self.pi_indices[state_idx]

        # Epsilon-greedy: explore randomly, or follow policy if available
        if explore and random.random() < exploration_prob:
            return random.choice(self.actions)
        if policy_idx == -1:
            return random.choice(self.actions)
        return self.actions[policy_idx]

    def incorporate_feedback(self, state, action, reward, next_state, terminal):
        state_idx = int(self.state_to_index(state))
        matches = np.where(self.actions_array == action)[0]
        action_idx = int(matches[0])
        next_idx = int(self.state_to_index(next_state))

        self.transition_counts[state_idx, action_idx, next_idx] += 1.0
        self.reward_sums[state_idx, action_idx, next_idx] += reward
        self.valid_actions[state_idx, action_idx] = True

        if self.num_iters > 0 and self.num_iters % self.calc_val_iter_every == 0:
            # Normalize transition counts into probabilities
            # Sum counts over next states for each (state, action) pair
            count_sums = self.transition_counts.sum(axis=2, keepdims=True)

            # Avoid division by zero for unvisited (state, action) pairs
            safe_sums = np.where(count_sums > 0, count_sums, 1.0)

            # T(s,a,s') = count(s,a,s') / sum_s' count(s,a,s')
            transitions = self.transition_counts / safe_sums

            # R(s,a,s') = total_reward(s,a,s') / count(s,a,s')
            safe_counts = np.where(self.transition_counts > 0, self.transition_counts, 1.0)
            rewards = self.reward_sums / safe_counts

            # Run value iteration on the estimated model
            self.pi_actions = value_iteration(
                transitions,
                rewards,
                self.discount,
                valid_actions=self.valid_actions,
                state_ids=self.state_ids,
                action_ids=self.actions_array,
            )

            self._sync_policy_indices()

In [ ]:
# Test ModelBasedMonteCarlo on the DiscreteGymMDP (Mountain Car)
from util_rl import DiscreteGymMDP, simulate

mdp = DiscreteGymMDP("MountainCar-v0", feature_bins=10, discount=0.99)
rl = ModelBasedMonteCarlo(
    actions=mdp.actions,
    discount=mdp.discount,
    num_states=mdp.num_states,
    state_to_index=mdp.state_to_index,
    index_to_state=mdp.index_to_state,
    calc_val_iter_every=10000,
    exploration_prob=0.2,
)

print("Training ModelBasedMonteCarlo...")
train_rewards = simulate(mdp, rl, num_trials=200, train=True, verbose=True)
print(f"\nTraining avg reward (last 50): {np.mean(train_rewards[-50:]):.2f}")

print("\nEvaluating learned policy...")
test_rewards = simulate(mdp, rl, num_trials=20, train=False, verbose=True)
print(f"Test avg reward: {np.mean(test_rewards):.2f}")

### Training with `uv run train.py --agent value-iteration`

Three independent training runs of 1000 episodes each, with value iteration recomputed every 10,000 steps.

![Training curves](vi_training_curves.png)

### Discussion

**Performance:** All three trials show a consistent pattern:
- Episodes 0-100: Random exploration phase (~-103 reward), as `exploration_prob` is forced to 1.0 for the first 20,000 steps
- Episodes 100-200: Sharp improvement to ~-93 as the first value iteration runs produce a useful policy from the accumulated transition model
- Episodes 200+: Plateaus around -93, with test performance reaching ~-90

The agent improves significantly from the naive baseline (-103) but plateaus well short of the optimal. The car never actually reaches the goal (all episodes hit the 200-step limit) but it does learn to oscillate closer to the goal position and with higher velocity.

**When model-based value iteration performs poorly:**

1. **Large/continuous state spaces**: The transition tensor is O(|S|^2 * |A|), which becomes intractable for high-dimensional problems. Even with discretization, fine bins cause the state space to explode.

2. **Insufficient exploration**: With a discrete model, many (s, a, s') transitions may never be observed, leading to an inaccurate model. The policy is only as good as the estimated transitions.

3. **Stochastic environments with many possible next states**: The model needs many samples per (s, a) pair to get accurate transition probabilities, which is sample-inefficient when transitions are highly stochastic.

4. **Non-stationary environments**: The model assumes fixed transition dynamics. If the environment changes over time, the accumulated counts become misleading.

5. **Discretization artifacts**: Binning continuous states loses information. Two nearby continuous states that map to different bins are treated as completely unrelated, while states at opposite edges of the same bin are treated as identical.

## Problem 4a: Tabular Q-Learning

In [ ]:
class TabularQLearning(util.RLAlgorithm):
    def __init__(
            self,
            actions: List,
            discount: float,
            num_states: int,
            state_to_index: Callable,
            exploration_prob: float = 0.2,
            initial_q: float = 0,
        ):
        self.actions = list(actions)
        self.actions_array = np.array(self.actions, dtype=object)
        self.discount = discount
        self.num_states = int(num_states)
        self.state_to_index = state_to_index
        self.exploration_prob = exploration_prob
        self.q = np.full((self.num_states, len(self.actions)), initial_q, dtype=np.float64)
        self.num_iters = 0

    def get_action(self, state, explore: bool = True):
        if explore:
            self.num_iters += 1
        exploration_prob = self.exploration_prob
        if self.num_iters < 2e4:  # explore
            exploration_prob = 1.0
        elif self.num_iters > 1e5:  # Lower the exploration probability by a logarithmic factor.
            exploration_prob = exploration_prob / math.log(self.num_iters - 100000 + 1)
        state_idx = int(self.state_to_index(state))
        if explore and random.random() < exploration_prob:
            return random.choice(self.actions)
        best_action_idx = int(np.argmax(self.q[state_idx]))
        return self.actions[best_action_idx]

    def get_step_size(self) -> float:
        return 0.1

    def incorporate_feedback(self, state, action, reward, next_state, terminal):
        state_idx = int(self.state_to_index(state))
        matches = np.where(self.actions_array == action)[0]
        action_idx = int(matches[0])
        eta = self.get_step_size()
        if terminal:
            v_next = 0.0
        else:
            next_idx = int(self.state_to_index(next_state))
            v_next = np.max(self.q[next_idx])
        target = reward + self.discount * v_next
        self.q[state_idx, action_idx] += eta * (target - self.q[state_idx, action_idx])